In [1]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

In [3]:
df=pd.read_csv('/content/ticket_classification.csv')
df=df.dropna()

In [4]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove BOTH actual newlines and literal "\n" text strings
    text = text.replace("\n", " ").replace("\\n", " ").replace("\r", " ")

    # 3. Isolates punctuation marks with spaces
    text = re.sub(r"([.,!?():\"'])", r" \1 ", text)

    # 4. Remove extra spaces and squash down to a single clean space
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
df['text'] = "Subject: "+df['subject'].apply(clean_text)+" Description: "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 Subject: account disruption Description: dear customer support team , i am writing to report a significant problem with the centralized account management portal , which currently appears to be offline . this outage is blocking access to account settings , leading to substantial inconvenience . i have attempted to log in multiple times using different browsers and devices , but the issue persists . could you please provide an update 

In [6]:
# 2. Encode Target
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['queue'])
num_classes = len(label_encoder.classes_)
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [7]:
id2label = {
    i: label
    for i, label in enumerate(label_encoder.classes_)
}

with open("id2label.json", "w") as f:
    json.dump(id2label, f)

In [ ]:
# # -----------------------------------
# # Compute class weights
# # -----------------------------------

# class_weights = compute_class_weight(

#     class_weight="balanced",

#     classes=np.unique(df["label"]),

#     y=df["label"]
# )

# # Convert to tensor

# class_weights = torch.tensor(
#     class_weights,
#     dtype=torch.float
# )

# print(class_weights)

In [8]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=128
    )

In [11]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist(),
    "label": val_labels.tolist()
})

In [12]:
train_dataset = train_dataset.map(tokenize_function, batched=True)

val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/19693 [00:00<?, ? examples/s]

Map:   0%|          | 0/4924 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="../saved_model/checkpoints",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=15,

    weight_decay=0.05,

    warmup_ratio=0.2,

    logging_dir="../saved_model/logs",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [21]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
# # -----------------------------------
# # Custom Weighted Trainer
# # -----------------------------------

# class WeightedTrainer(Trainer):

#     def compute_loss(
#         self,
#         model,
#         inputs,
#         return_outputs=False,
#         num_items_in_batch=None
#     ):

#         # Extract labels

#         labels = inputs.pop("labels")

#         # Forward pass

#         outputs = model(**inputs)

#         logits = outputs.get("logits")

#         # Weighted CrossEntropy Loss

#         loss_fn = nn.CrossEntropyLoss(

#             weight=class_weights.to(model.device)
#         )

#         # Compute loss

#         loss = loss_fn(

#             logits.view(
#                 -1,
#                 self.model.config.num_labels
#             ),

#             labels.view(-1)
#         )

#         return (
#             (loss, outputs)
#             if return_outputs
#             else loss
#         )

In [26]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# trainer = WeightedTrainer(

#     model=model,

#     args=training_args,

#     train_dataset=train_dataset,

#     eval_dataset=val_dataset,

#     processing_class=tokenizer,

#     data_collator=data_collator,

#     compute_metrics=compute_metrics
# )

In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.296587,1.327140,0.690089,0.687139
2,0.331742,1.487740,0.684403,0.682677
3,0.344878,1.814866,0.659626,0.652793


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7386, training_loss=0.32131106027767903, metrics={'train_runtime': 760.4718, 'train_samples_per_second': 388.436, 'train_steps_per_second': 48.562, 'total_flos': 1843719684320400.0, 'train_loss': 0.32131106027767903, 'epoch': 3.0})

In [28]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.344878,1.327140,3,0.690089,0.687139


{'eval_loss': 1.3271404504776,
 'eval_accuracy': 0.690089358245329,
 'eval_f1': 0.687139070389138}

In [29]:
model.save_pretrained("distilbert_classifier")

tokenizer.save_pretrained("distilbert_tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_tokenizer/tokenizer_config.json',
 'distilbert_tokenizer/tokenizer.json')